# ManyBodyQutip — how this repo builds and uses its Hamiltonians

This notebook is a project-specific companion to
[emacosta95/ManyBodySystemQutip](https://github.com/emacosta95/ManyBodySystemQutip)
(imported here as `ManyBodyQutip`, pinned in `requirements.txt`). It is **not**
a general API reference for the library — it explains exactly how *this*
repository (`Magic4Annealing`) uses it to build the driver and target
Hamiltonians for the quantum-annealing optimal-control study, and why the
Z2 symmetry-sector projection (`src/utils.py`) matters for that pipeline.

**Reading order:**
1. `SpinOperator` — the building block: how a Pauli string term becomes a
   sparse many-body matrix.
2. `SpinHamiltonian` — the higher-level convenience wrapper (available in
   the library; not currently used by this repo's scripts, which build
   Hamiltonians directly from `SpinOperator` sums instead — see below).
3. `src/annealing_utils.py` — the two functions this project actually
   calls: `get_driver_hamiltonian` and `get_longitudinal_hamiltonian`.
4. The Z2 symmetric-sector projection (`src/utils.py`) — why the naive
   `Sector` class is wrong, why `Z2SymmetricSector` is correct, and how
   the two integrate with the optimal-control models
   (`SparseGRAPEModel` / `SchedulerModel`).


## 0. Setup

Adjust `REPO_ROOT` to wherever you've cloned `Magic4Annealing` locally.
`ManyBodyQutip` should already be installed per `requirements.txt`
(`pip install git+https://github.com/emacosta95/ManyBodySystemQutip.git`).


In [1]:
import sys, os

REPO_ROOT = os.path.abspath(".")   # run this notebook from the repo root
sys.path.insert(0, REPO_ROOT)

import numpy as np
from scipy.sparse.linalg import eigsh

from ManyBodyQutip.qutip_class import SpinOperator, SpinHamiltonian

np.set_printoptions(precision=2, suppress=True)


## 1. `SpinOperator` — the building block

`SpinOperator` turns a list of Pauli-string terms into a single sparse
many-body operator on `size` qubits. Each term is specified as a flat tuple
alternating `(direction, site_index, direction, site_index, ...)`, with a
matching coupling constant. Sites not mentioned in a term are implicitly
padded with identity.

Available single-site directions (see `_local_obs_dict` in
`qutip_class.py`): `'x'`, `'y'`, `'z'` (Pauli matrices), `'+'`/`'-'`
(raising/lowering), `'id'` (identity), plus two projectors `'qz'`
(onto \|1⟩) and `'qx'` (onto \|+⟩).

Below: a single ZZ bond between qubits 0 and 1 on a 3-qubit chain — note
sites 0,1 get ±1 eigenvalues of Z⊗Z, site 2 is padded with identity, so the
diagonal pattern repeats every 2 basis states.


In [2]:
n = 3
zz_01 = SpinOperator(index=[("z", 0, "z", 1)], coupling=[1.0], size=n)
print(zz_01)                      # human-readable description
print(zz_01.qutip_op.full().real)  # dense matrix (fine for small n only)


(couplings, operator) -> 
 ( 1.0 ,  ('z', 0, 'z', 1) ) 
 

[[ 1.  0.  0.  0.  0.  0.  0.  0.]
 [ 0.  1.  0.  0.  0.  0.  0.  0.]
 [ 0.  0. -1.  0.  0.  0.  0.  0.]
 [ 0.  0.  0. -1.  0.  0.  0.  0.]
 [ 0.  0.  0.  0. -1.  0.  0.  0.]
 [ 0.  0.  0.  0.  0. -1.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  1.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.  1.]]


In [3]:
# A uniform transverse field -sum_i X_i (this is exactly this project's
# driver Hamiltonian ansatz — coupling=-1 on every site, see Section 3).
x_field = SpinOperator(
    index=[("x", i) for i in range(n)],
    coupling=[-1] * n,
    size=n,
)
print(x_field)
print("diag (should be all zero -- X has no diagonal part):",
      np.diag(x_field.qutip_op.full().real))
print("row 0 (population transfer amplitudes):",
      x_field.qutip_op.full().real[0])


(couplings, operator) -> 
 ( -1 ,  ('x', 0) ) 
 ( -1 ,  ('x', 1) ) 
 ( -1 ,  ('x', 2) ) 
 

diag (should be all zero -- X has no diagonal part): [0. 0. 0. 0. 0. 0. 0. 0.]
row 0 (population transfer amplitudes): [ 0. -1. -1.  0. -1.  0.  0.  0.]


Multiple terms in a single `SpinOperator` call are summed automatically —
e.g. `index=[("z",0,"z",1), ("z",1,"z",2)]` with two couplings gives a
2-bond ZZ chain in one object. This is exactly the pattern
`get_longitudinal_hamiltonian` below uses, just looped one bond at a time
and accumulated with `+=` instead of passed as one multi-term list — both
are equivalent, this repo just builds it bond-by-bond.


## 2. `SpinHamiltonian` — available, but not used by this project

The library also provides `SpinHamiltonian`, a convenience class that
builds a full transverse-field-Ising-style Hamiltonian (couplings +
external fields, with periodic-boundary support) from arrays of coupling
values/directions in one call, without you having to loop over bonds
yourself.

This project's own `src/annealing_utils.py` does **not** use
`SpinHamiltonian` — it builds the driver and target Hamiltonians directly
from `SpinOperator` sums instead (Section 3), presumably for finer control
over the exact coupling structure (e.g. an arbitrary frustrated-ring
adjacency matrix `jij`, not just a uniform/PBC chain). Shown here for
awareness in case it's useful for a future model variant — e.g. a quick
uniform transverse-field Ising chain:


In [4]:
demo_ham = SpinHamiltonian(
    direction_couplings=[("z", "z")],
    coupling_values=[1.0],
    field_directions=["x"],
    field_values=[-1.0],
    pbc=False,
    size=3,
)
print(demo_ham.qutip_op.full().real)


Hermitian Check positive! well done! 

[[ 2. -2. -2.  0. -2.  0.  0.  0.]
 [-2.  0.  0. -2.  0. -2.  0.  0.]
 [-2.  0. -2. -2.  0.  0. -2.  0.]
 [ 0. -2. -2.  0.  0.  0.  0. -2.]
 [-2.  0.  0.  0.  0. -2. -2.  0.]
 [ 0. -2.  0.  0. -2. -2.  0. -2.]
 [ 0.  0. -2.  0. -2.  0.  0. -2.]
 [ 0.  0.  0. -2.  0. -2. -2.  2.]]


## 3. How *this* repo builds driver/target Hamiltonians

`src/annealing_utils.py` has two functions used throughout the study
scripts (`study_1d_ising.py`, `study_avoided_crossing.py`,
`measure_magic_optimal_control.py`, ...):

```python
def get_driver_hamiltonian(nqubits):
    hamiltonian_x = 0.0
    for i in range(nqubits):
        hamiltonian_x += SpinOperator(
            [("x", i)], coupling=[-1], size=nqubits, verbose=1
        ).qutip_op
    return hamiltonian_x.data.as_scipy()


def get_longitudinal_hamiltonian(jij, hz=None):
    n_qubits = jij.shape[0]
    hamiltonian_zz = 0.0
    for i in range(n_qubits):
        for j in range(i + 1, n_qubits):
            hamiltonian_zz += SpinOperator(
                [("z", i, "z", j)], coupling=[jij[i, j]], size=n_qubits, verbose=1
            ).qutip_op
    hamiltonian_z = 0.0
    for i in range(n_qubits):
        hamiltonian_z += SpinOperator(
            [("z", i)], coupling=[hz[i]], size=n_qubits, verbose=1
        ).qutip_op
    return (hamiltonian_zz + hamiltonian_z).data.as_scipy()
```

Three things worth calling out explicitly for anyone extending this:

- **`.qutip_op.data.as_scipy()`** is the bridge between ManyBodyQutip
  (which works with `qutip.Qobj` internally) and the rest of this repo's
  optimal-control code (`SparseGRAPEModel`, `SchedulerModel`, ...), which
  expects plain `scipy.sparse` matrices, not `Qobj`s. Every Hamiltonian
  that leaves `annealing_utils.py` has already been converted this way.
- **`get_driver_hamiltonian`** hardcodes `coupling=-1`, i.e. the driver is
  always exactly $H_{\text{driver}} = -\sum_i X_i$ (uniform transverse
  field, negative sign) — not configurable per call.
- **`get_longitudinal_hamiltonian(jij, hz)`** builds
  $H_{\text{target}} = \sum_{i<j} J_{ij} Z_i Z_j + \sum_i h_i Z_i$ from an
  arbitrary adjacency matrix `jij` — this is what lets the project target
  e.g. the frustrated-ring instance (`jij` = ring adjacency with your
  chosen couplings) without changing any code, just the `jij` array passed
  in from the study script.


In [5]:
from src.annealing_utils import get_driver_hamiltonian, get_longitudinal_hamiltonian

n = 5
# frustrated antiferromagnetic ring: J_{i,i+1} = 1 for all bonds (mod n)
jij = np.zeros((n, n))
for i in range(n):
    jij[i, (i + 1) % n] = 1.0
    jij[(i + 1) % n, i] = 1.0

H_driver = get_driver_hamiltonian(n)
H_target = get_longitudinal_hamiltonian(jij)

print("H_driver:", H_driver.shape, type(H_driver))
print("H_target:", H_target.shape, type(H_target))

H_full = (H_driver + H_target).toarray()
print("Hermitian check:", np.allclose(H_full, H_full.conj().T))


H_driver: (32, 32) <class 'scipy.sparse._csr.csr_matrix'>
H_target: (32, 32) <class 'scipy.sparse._csr.csr_matrix'>
Hermitian check: True


## 4. The Z2 symmetric-sector projection (`src/utils.py`)

Both Hamiltonians above commute with the **global spin flip**
$\Pi = \prod_i X_i$ (every term is either a product of an even number of
$Z$'s, or a single $X$ — both commute with flipping every qubit). That
means $\Pi$ is a conserved symmetry, the Hilbert space splits into a $+1$
and a $-1$ eigenspace of $\Pi$ that never mix under time evolution, and
the physically relevant dynamics for this project lives entirely in the
$+1$ sector — because the initial state, the uniform superposition
$|+\rangle^{\otimes n}$, is itself a $+1$ eigenstate of $\Pi$ (true for any
permutation-symmetric state).

This halves the effective Hilbert space dimension used by every
optimal-control run — a real, significant speedup, not just a
bookkeeping convenience. But getting the projection right matters:

### 4a. The broken version: `Sector` (naive index truncation)

`Sector` just keeps computational-basis indices `0 .. 2^(n-1)-1` and drops
the rest — cheap, but **wrong** in general: it only gives correct
eigenvalues if $H$ happens to already be block-diagonal under that raw
index split, which isn't guaranteed just because every $\Pi$-degenerate
pair $(x, \bar x)$ has one representative below $2^{n-1}$.


In [6]:
from src.utils import Sector, Z2SymmetricSector

S = Sector(nqubits=n)
H_naive = S.project(H_driver + H_target)

evals_full = np.sort(np.linalg.eigvalsh(H_full))
evals_naive = np.sort(np.linalg.eigvalsh(H_naive.toarray()))

print("lowest 5 full eigenvalues:  ", evals_full[:5])
print("lowest 5 naive-sector evals:", evals_naive[:5])
print("max abs diff (low-lying 5):", np.max(np.abs(evals_full[:5] - evals_naive[:5])))


Sector: 16 states out of 32
lowest 5 full eigenvalues:   [-6.16 -5.24 -5.24 -3.8  -3.8 ]
lowest 5 naive-sector evals: [-5.46 -4.37 -3.41 -2.66 -2.19]
max abs diff (low-lying 5): 1.8288367299486619


That's an O(1) discrepancy — not a rounding error, a genuinely wrong
spectrum. Any GRAPE/optimal-control run built on `Sector` would be
optimizing against the wrong Hamiltonian entirely.

### 4b. The correct version: `Z2SymmetricSector`

`Z2SymmetricSector` builds the projection properly, as an explicit
isometry $U$ (shape `(2^(n-1), 2^n)`) onto symmetric/antisymmetric basis
combinations

$$|\text{sym}_x\rangle = \frac{|x\rangle + |\bar x\rangle}{\sqrt2}, \qquad
  |\text{asym}_x\rangle = \frac{|x\rangle - |\bar x\rangle}{\sqrt2}$$

for $x < 2^{n-1}$, then projects via $H_{\text{sector}} = U H U^\dagger$.
Combining the `sign=+1` and `sign=-1` sectors must reproduce the *exact*
full spectrum (to machine precision) — that's the correctness check:


In [7]:
Zp = Z2SymmetricSector(nqubits=n, sign=+1)
Zm = Z2SymmetricSector(nqubits=n, sign=-1)

Hp = Zp.project(H_driver + H_target)
Hm = Zm.project(H_driver + H_target)

evals_combined = np.sort(np.concatenate([
    np.linalg.eigvalsh(Hp.toarray()),
    np.linalg.eigvalsh(Hm.toarray()),
]))

print("max abs diff, full vs combined ±1 sectors:",
      np.max(np.abs(evals_full - evals_combined)))


Z2SymmetricSector: 16 states out of 32 (sign=+1)
Z2SymmetricSector: 16 states out of 32 (sign=-1)
max abs diff, full vs combined ±1 sectors: 5.773159728050814e-15


Machine precision (~1e-14 to 1e-15) — this is the projection this project
actually relies on.

### 4c. Why sign=+1 specifically, and the project/lift round trip

The uniform superposition $|+\rangle^{\otimes n}$ (this project's initial
state for the annealing protocol) lies *entirely* in the `sign=+1` sector:


In [8]:
psi_full = np.ones(2**n, dtype=complex) / np.sqrt(2**n)
psi_plus = Zp.project(psi_full)
psi_minus = Zm.project(psi_full)

print("norm in +1 sector:", np.linalg.norm(psi_plus))
print("norm in -1 sector:", np.linalg.norm(psi_minus))


norm in +1 sector: 1.0
norm in -1 sector: 0.0


So for this project's pipeline, **everything** — both Hamiltonians *and*
the initial state — gets projected with `sign=+1`, and the entire
GRAPE / linear-ramp / gradient-free optimization runs inside that reduced
`(2^(n-1), 2^(n-1))`-dimensional space:

```python
PS = Z2SymmetricSector(nqubits=n, sign=+1)
driver_hamiltonian_s = PS.project(driver_hamiltonian)
target_hamiltonian_s = PS.project(target_hamiltonian)
psi_init_s = PS.project(psi_init_full)   # NOT a fresh uniform superposition
                                           # over the truncated basis --
                                           # the actual physical state,
                                           # projected correctly.

model = SparseGRAPEModel(
    initial_state=psi_init_s,
    target_hamiltonian=target_hamiltonian_s,
    initial_hamiltonian=driver_hamiltonian_s,
    reference_hamiltonian=target_hamiltonian_s,
    ...
)
```

The one place you must **leave** the sector: any observable that's a
functional of the *full* $2^n$-dimensional wavefunction and doesn't
respect the sector structure the same way a Hamiltonian expectation value
does — e.g. stabilizer Rényi entropy (`SREJax`) or entanglement entropy
(`EntanglementEntropy`) in this project. For those, `.lift()` embeds the
sector-space state back into the full Hilbert space first:

```python
psi_full_t = PS.lift(psi_sector_t)
magic_t = sre(psi_full_t)
ee_t = entanglement_entropy.von_neumann(psi_full_t)
```


In [9]:
# project/lift round trip sanity check
psi_lifted = Zp.lift(psi_plus)
print("round-trip matches original:", np.allclose(psi_lifted, psi_full))


round-trip matches original: True


## 5. Full pipeline, summarized

```
ManyBodyQutip.SpinOperator          (Pauli-string terms -> qutip.Qobj)
        |  .qutip_op.data.as_scipy()
        v
src/annealing_utils.py              (get_driver_hamiltonian, get_longitudinal_hamiltonian
        |                            -> plain scipy.sparse matrices)
        v
src/utils.py Z2SymmetricSector      (project driver, target, AND initial_state
        |                            with sign=+1 -> reduced (2^(n-1)) space)
        v
src/sparse_grape_method.py          (SparseGRAPEModel / SparseGRAPETrainer)
  or src/schedule_utils.py          (SchedulerModel / SchedulerTrainer)
        |                            -> optimized schedule + psi(t) in
        |                               SECTOR space
        v
Z2SymmetricSector.lift()            (back to full 2^n space, only when needed
        |                            for SRE / entanglement entropy)
        v
src/utils.py SREJax, EntanglementEntropy
```

**Gotcha worth remembering** (also noted in `src/annealing_utils.py`
usage elsewhere in this project): when extracting a dominant computational
basis state from an `eigsh` eigenvector, find it via
`np.argmax(np.abs(vec)**2)`, not by assuming any particular index
ordering — `eigsh` does not return eigenvectors sorted by their
computational-basis overlap, only (optionally) by eigenvalue.
